# IOAI — 2025 Team Selection Day1 Cv Image Restoration (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day1-cv-image-restoration'
for f in ['train_clean.npy','train_filtered.npy','val_filtered.npy']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train_clean.npy','train_filtered.npy','val_filtered.npy'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 1 — Image Restoration (Kazakhstan IOAI Team Selection 2025)

> **Kazakhstan – Team Selection 2025 · Day 1 (Computer Vision)**

Each 128×128 image was corrupted by a **2×2 per-channel filter**: inside every 2×2 block each of the four pixels keeps **only one** colour channel (R, G or B) and the other two are zeroed. Your job is to **restore the original image**. Score = **PSNR** (higher is better) of the restored validation images against the hidden clean originals. This baseline trains a small U-Net mapping *filtered → clean* and saves `val_pred.npy`.

*Real-data notice:* this uses the **original competition data** from the public up-solving mirror `kaggle.com/competitions/up-solving-tst-day-1` (a bundled subset of 800 train / 200 val pairs). The hidden test set is graded on Kaggle; here we self-grade on a held-out validation split via PSNR.

In [ ]:
import os, numpy as np, torch
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day1_CV_Image_Restoration"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
DATA = _find("train_clean.npy")
Xtr_c = np.load(os.path.join(DATA,"train_clean.npy"))      # (N,128,128,3) uint8  (clean targets)
Xtr_f = np.load(os.path.join(DATA,"train_filtered.npy"))   # (N,128,128,3) uint8  (filtered inputs)
Xva_f = np.load(os.path.join(DATA,"val_filtered.npy"))     # (M,128,128,3) uint8  (val inputs)
print("train", Xtr_c.shape, "val", Xva_f.shape, "device", device)

def detect_filter(f):
    """Each 2x2 block keeps exactly one colour channel per pixel. Return per-position kept channel (4 ids 0..2)."""
    m = (f > 0)
    key = []
    for py in range(2):
        for px in range(2):
            key.append(int(m[py::2, px::2, :].reshape(-1, 3).mean(0).argmax()))
    return key  # e.g. [2,0,0,1]


## Dataset — filtered input → clean target

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

def to_t(x): return torch.tensor(x.astype(np.float32)/255.).permute(0,3,1,2).contiguous()
tr = DataLoader(TensorDataset(to_t(Xtr_f), to_t(Xtr_c)), batch_size=32, shuffle=True)


## Compact U-Net (baseline)

In [ ]:
def cbr(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                                   nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
class UNet(nn.Module):
    def __init__(s):
        super().__init__()
        s.d1=cbr(3,32); s.d2=cbr(32,64); s.d3=cbr(64,128); s.pool=nn.MaxPool2d(2); s.bott=cbr(128,256)
        s.u3=nn.ConvTranspose2d(256,128,2,2); s.c3=cbr(256,128)
        s.u2=nn.ConvTranspose2d(128,64,2,2);  s.c2=cbr(128,64)
        s.u1=nn.ConvTranspose2d(64,32,2,2);   s.c1=cbr(64,32)
        s.out=nn.Conv2d(32,3,1)
    def forward(s,x):
        e1=s.d1(x); e2=s.d2(s.pool(e1)); e3=s.d3(s.pool(e2)); b=s.bott(s.pool(e3))
        d=s.c3(torch.cat([s.u3(b),e3],1)); d=s.c2(torch.cat([s.u2(d),e2],1)); d=s.c1(torch.cat([s.u1(d),e1],1))
        return torch.sigmoid(s.out(d))
model=UNet().to(device); opt=torch.optim.Adam(model.parameters(),1e-3)


## Train (MSE)

In [ ]:
EPOCHS=10
for ep in range(EPOCHS):
    model.train(); tot=0
    for xb,yb in tr:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad(); out=model(xb); loss=F.mse_loss(out,yb); loss.backward(); opt.step()
        tot+=loss.item()*len(xb)
    print(f"epoch {ep+1}/{EPOCHS}  mse={tot/len(tr.dataset):.4f}")


## Predict validation → `val_pred.npy` (submit this)

In [ ]:
model.eval()
preds=[]
with torch.no_grad():
    for i in range(0,len(Xva_f),64):
        xb=to_t(Xva_f[i:i+64]).to(device)
        preds.append((model(xb).clamp(0,1).cpu().numpy()*255).round().astype(np.uint8))
val_pred=np.concatenate(preds).transpose(0,2,3,1)   # (M,128,128,3) uint8
np.save("val_pred.npy", val_pred)
print("saved val_pred.npy", val_pred.shape)

# quick local PSNR (for your own reference; official grading uses the hidden clean val set)
try:
    gt=np.load(_find("val_clean.npy")+"/val_clean.npy") if os.path.exists(_find("val_clean.npy")+"/val_clean.npy") else None
except Exception: gt=None


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['val_pred.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)